In [ ]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor
import importlib.util

# Find repo root robustly whether execution starts in repo root or in code/
HERE = Path.cwd()

if (HERE / "data").exists() and (HERE / "results").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "data").exists() and (HERE.parent / "results").exists():
    REPO_ROOT = HERE.parent
else:
    raise FileNotFoundError(
        f"Could not locate repo root from working directory: {HERE}"
    )

print("Working directory:", HERE)
print("Resolved repo root:", REPO_ROOT)
# -------------------------------------------------
# Paths
# -------------------------------------------------
DATA_PATH = REPO_ROOT / "data"
TERRAIN_PATH = DATA_PATH / "weightedTerrainData.csv"

INTERMEDIATE_DIR = REPO_ROOT / "results" / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

OUT_LONG = INTERMEDIATE_DIR / "table4_2017_lgbm_long.csv"
OUT_SUMM = INTERMEDIATE_DIR / "table4_2017_lgbm_summary.csv"

# -------------------------------------------------
# IDs
# -------------------------------------------------
TURBINE_IDS = list(range(1, 67))
TESTSET  =  [38,39,40,41,42,43,44] 
TRAINSET = [i for i in TURBINE_IDS if i not in TESTSET]

# -------------------------------------------------
# Terrain (cols 2:4, NO scaling)
# -------------------------------------------------
terrain_df = pd.read_csv(TERRAIN_PATH)
terrain_cols = terrain_df.columns[1:4]
terrain_mat = terrain_df.loc[:65, terrain_cols].values  # row i-1 = turbine i

# -------------------------------------------------
# Helpers
# -------------------------------------------------
def load_turbine_csv(tid, year):
    f = DATA_PATH / f"Turbine{tid}_{year}.csv"
    if not f.exists():
        return None
    df = pd.read_csv(f)
    for c in ["wind_speed", "temperature", "power"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def rmse(yhat, y):
    yhat = np.asarray(yhat, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(yhat) & np.isfinite(y)
    if not np.any(m):
        return np.nan
    return float(np.sqrt(np.mean((yhat[m] - y[m])**2)))

# -------------------------------------------------
# Cache 2017 data only
# -------------------------------------------------
data_cache = {}
for tid in TURBINE_IDS:
    data_cache[(tid, 2017)] = load_turbine_csv(tid, 2017)

# -------------------------------------------------
# Build full 2017 training pool using TRAINSET only
# -------------------------------------------------
X2017_list, y2017_list, tid2017_list = [], [], []

for tid in TRAINSET:
    df = data_cache[(tid, 2017)]
    if df is None:
        continue

    df = df[["wind_speed", "temperature", "power"]].dropna()
    if df.empty:
        continue

    X = df[["wind_speed", "temperature"]].values
    y = df["power"].values
    t = np.full(len(y), tid, dtype=int)

    X2017_list.append(X)
    y2017_list.append(y)
    tid2017_list.append(t)

if len(X2017_list) == 0:
    raise ValueError("No valid training data found in TRAINSET.")

X2017 = np.vstack(X2017_list)
y2017 = np.concatenate(y2017_list)
tid2017 = np.concatenate(tid2017_list)

S2017 = terrain_mat[tid2017 - 1]

# Fixed training set from TRAINSET
X_train = X2017
y_train = y2017
S_train = S2017

# -------------------------------------------------
# Fixed LightGBM model
# -------------------------------------------------
seed = 42

model = LGBMRegressor(
    objective="regression",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=seed,
    n_jobs=-1
)
model.fit(X_train, y_train)

def fit_lgbm(X_train, y_train, X_test, seed):
    return model.predict(X_test)

# -------------------------------------------------
# Main loop: test on 2017 only, using one fixed training set
# -------------------------------------------------
results = []

for i in TESTSET:
    print(f"Evaluating Turbine {i}, Year 2017")

    df_test = data_cache[(i, 2017)]
    if df_test is None:
        continue

    df_test = df_test[["wind_speed", "temperature", "power"]].dropna()
    if df_test.empty:
        continue

    X_test = df_test[["wind_speed", "temperature"]].values
    y_test = df_test["power"].values

    S_test = np.tile(terrain_mat[i - 1], (len(X_test), 1))

    t0 = time.time()
    pred_x = fit_lgbm(X_train, y_train, X_test, seed=i)
    t1 = time.time()

    results.append({
        "Method": "LGBM(x)",
        "Turbine": i,
        "Year": 2017,
        "RMSE": rmse(pred_x, y_test),
        "Runtime": round(t1 - t0, 4)
    })

# -------------------------------------------------
# Save outputs
# -------------------------------------------------
results_long = pd.DataFrame(results)
results_long.to_csv(OUT_LONG, index=False)

summary = (
    results_long
    .groupby(["Method", "Year"], as_index=False)
    .agg(RMSE=("RMSE", "mean"),
         Runtime=("Runtime", "mean"))
)

summary_wide = summary.pivot(index="Method", columns="Year")
summary_wide.columns = [f"{a}{b}" for a, b in summary_wide.columns]
summary_wide = summary_wide.reset_index()

print("Saved results:")
print(" -", OUT_LONG)
print(" -", OUT_SUMM)

summary_wide.to_csv(OUT_SUMM, index=False)


# ---- Update final results.csv : Table 4, Multi-layer NN row ----
from pathlib import Path
import importlib.util

upd_path = REPO_ROOT / "code" / "update_final_results.py"
spec = importlib.util.spec_from_file_location("update_final_results", upd_path)
upd_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(upd_mod)
update_final_results = upd_mod.update_final_results

row_x = summary_wide.loc[summary_wide["Method"] == "LGBM(x)", "RMSE2017"]
xgb_table4_rmse = row_x.iloc[0] if len(row_x) > 0 else np.nan

update_final_results(method="XGBoost", table_id="Table 4", rmse=xgb_table4_rmse, version="x")

# -------------------------------------------------
# XGBoost(x+s): retrain with terrain concatenated
# -------------------------------------------------
Xs_train = np.hstack([X_train, S_train])

model_xs = LGBMRegressor(
    objective="regression", n_estimators=200, learning_rate=0.1,
    max_depth=8, subsample=0.8, colsample_bytree=0.8,
    random_state=seed, n_jobs=-1
)
model_xs.fit(Xs_train, y_train)

results_xs = []
for i in TESTSET:
    df_test = data_cache[(i, 2017)]
    if df_test is None: continue
    df_test = df_test[["wind_speed","temperature","power"]].dropna()
    if df_test.empty: continue
    X_test = df_test[["wind_speed","temperature"]].values
    y_test = df_test["power"].values
    S_test = np.tile(terrain_mat[i-1], (len(X_test), 1))
    Xs_test = np.hstack([X_test, S_test])
    t0 = time.time()
    pred_xs = model_xs.predict(Xs_test)
    t1 = time.time()
    results_xs.append({"Method":"LGBM(x+s)","Turbine":i,"Year":2017,
                        "RMSE":rmse(pred_xs, y_test),"Runtime":round(t1-t0,4)})

results_long_xs = pd.DataFrame(results_xs)
results_long_xs.to_csv(INTERMEDIATE_DIR / "table4_2017_lgbm_xs_long.csv", index=False)

xgb_xs_rmse = results_long_xs["RMSE"].mean() if not results_long_xs.empty else np.nan

update_final_results(method="XGBoost", table_id="Table 4", rmse=xgb_xs_rmse, version="xs")
